# Data Science Lab on Smart Cities

## Decoding the Intersection between Ridesourcing Dependency and Socio-Demographic Vulnerability in the Chicago FUA

Research question: Is ridesourcing acting as a mobility equalizer or as a costly substitute for inadequate public transport in vulnerable peripheral areas of Chicago?

**Analysis flow**

1. Data sources
2. Transport Usage Index (TUI)
3. Socio-Demographic Vulnerability Index (SDVI)
4. TUI vs vulnerability (correlations and maps)
5. Weighted TUI (Financial Burden Index)
6. Ridesharing travel time to the Loop
7. Transit Index
8. OLS
9. SEM/SLAG

## 1. Data Sources

- CMAP Community Data Snapshots — [ArcGIS FeatureServer](https://services5.arcgis.com/LcMXE3TFhi1BSaCY/arcgis/rest/services/CommunityDataSnapshots_2015_2025_gdb/FeatureServer) · [Data Hub](https://datahub.cmap.illinois.gov/search?tags=community%2520data%2520snapshots)
- Transportation Network Providers trips (Chicago Data Portal): [2018–2022](https://data.cityofchicago.org/Transportation/Transportation-Network-Providers-Trips-2018-2022-/m6dm-c72p/data_preview) · [2023–2024](https://data.cityofchicago.org/Transportation/Transportation-Network-Providers-Trips-2023-2024-/n26f-ihde/data_preview)
- Hardship Index and CCVI (Chicago Data Portal)
- Community Area boundaries — [Chicago Data Portal](https://data.cityofchicago.org/Facilities-Geographic-Boundaries/Boundaries-Community-Areas-Map/cauq-8yn6)


In [1]:
# Install dependencies (skipped when already satisfied)
import subprocess
import sys

# PyPI backport shadows stdlib importlib on Python 3.13 and breaks pip
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "importlib"],
    capture_output=True,
)

%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import importlib

import matplotlib.pyplot as plt
import pandas as pd
import utils
import numpy as np
import statsmodels.formula.api as smf
from libpysal.weights import Queen
from esda.moran import Moran, Moran_Local
from splot.esda import moran_scatterplot

importlib.reload(utils)

from utils import (
    DEFAULT_AVG_TRIP_COST_USD,
    build_analysis_gdf,
    build_loop_travel_time_map,
    build_tui_map,
    build_vulnerability_indices,
    build_weighted_tui_map,
    compute_tui_correlations,
    compute_weighted_tui,
    load_all_cca_population,
    load_chicago_community_areas,
    load_vulnerability_data,
    plot_loop_travel_time_map,
    plot_tui_index,
    plot_tui_vulnerability_maps,
    plot_tui_vulnerability_scatter,
    plot_weighted_tui_map,
    save_chart,
    source_loop_travel_times,
    source_tnp_counts,
    source_tnp_fares,
    zscore,
    compute_moran,
    plot_moran_scatterplot,
    compute_lisa,
    plot_lisa_map, 
    compute_transit_accessibility,
    plot_transit_deficit,
    run_spatial_error_model,
    run_spatial_lag_model,
    moran_residuals_spatial_model,
)

/Users/francesca.negri/.pyenv/versions/3.13.5/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Transport Usage Index (TUI)

TNP Usage Intensity measures ridesourcing trips per 1,000 residents in Community Area $i$ during month $m$:

$$TUI_{i,m} = \frac{\text{TNP pickups}_{i,m}}{\text{population}_{i,m}} \times 1000$$

Higher values indicate more TNP activity per resident. The index reflects usage intensity, not dependency, unless combined with accessibility indicators.

In [ ]:
url_2023_2024 = "https://data.cityofchicago.org/resource/n26f-ihde.json"
source_tnp_counts(url_2023_2024, "tnp_counts_2023-2024")

In [ ]:
url_2018_2022 = "https://data.cityofchicago.org/resource/m6dm-c72p.json"
source_tnp_counts(url_2018_2022, "tnp_counts_2018-2022")

In [ ]:
tnp_counts_2018_2022 = pd.read_csv("data/tnp_counts_2018-2022.csv")
tnp_counts_2023_2024 = pd.read_csv("data/tnp_counts_2023-2024.csv")
tnp_counts = pd.concat([tnp_counts_2018_2022, tnp_counts_2023_2024], ignore_index=True)
tnp_counts["month"] = pd.to_datetime(tnp_counts["month"])
tnp_counts["year"] = tnp_counts["month"].dt.year
tnp_counts = tnp_counts.rename(columns={"pickup_community_area": "community_area"})

population = load_all_cca_population("data")
df = pd.merge(tnp_counts, population, on=["community_area", "year"], how="left")

In [ ]:
df["tui_index"] = (df["n_trips"] / df["population"]) * 1000

In [ ]:
df.to_csv("data/tui_index.csv", index=False)
df.head()

In [ ]:
tui = pd.read_csv("data/tui_index.csv", parse_dates=["month"])
chicago_community_areas = load_chicago_community_areas(
    "data/Boundaries_-_Community_Areas_20260607.geojson"
)

tui_map_2018 = build_tui_map(chicago_community_areas, tui, year=2018)
tui_map_2024 = build_tui_map(chicago_community_areas, tui, year=2024)

In [ ]:
ax = plot_tui_index(tui_map_2018, year=2018, add_labels=False)
save_chart(ax, "tui_2018")
plt.show()

In [ ]:
ax = plot_tui_index(tui_map_2024, year=2024, add_labels=False)
save_chart(ax, "tui_2024")
plt.show()

## 3. Socio-Demographic Vulnerability Index (SDVI)

We build two composite vulnerability indices ourselves by combining and z-standardizing source indicators. **CCVI** (Chicago COVID-19 Community Vulnerability Index) is instead an external indicator published by the Chicago Data Portal, which we use as an input and as an auxiliary health-related vulnerability measure.

- **SDVI** (Socio-Demographic Vulnerability Index): our index combining the **Hardship Index** and **per capita income** (income enters as disadvantage: lower income implies higher vulnerability). Defined as `mean(z(Hardship), z(-income))`. Higher SDVI means greater socio-demographic vulnerability.
- **HSVI** (baseline): our index combining the **Hardship Index** and the external **CCVI**. Defined as `mean(z(Hardship), z(CCVI))`.

Indices are computed in two steps: `load_vulnerability_data` retrieves and cleans the raw inputs, then `build_vulnerability_indices` constructs SDVI and HSVI.

Community Area 76 (O'Hare) is excluded from socio-demographic modelling.

In [ ]:
vulnerability_raw = load_vulnerability_data("data")
vulnerability = build_vulnerability_indices(vulnerability_raw)
analysis_gdf = build_analysis_gdf(tui_map_2024, vulnerability)

print(f"Community Areas in analysis (excl. O'Hare): {len(analysis_gdf)}")
analysis_gdf[[
    "community_area", "community_area_name", "tui_index",
    "hardship_index", "per_capita_income", "hsvi", "sdvi", "ccvi_score",
]].head()

### Morans I
Is ridesourcing intensity spatially clustered, or randomly distributed across Chicago?
If high-TUI areas are near other high-TUI areas, then TUI has a spatial structure.

In [ ]:
moran = compute_moran(analysis_gdf, "tui_index")
plot_moran_scatterplot(moran)

The Moran scatterplot shows a clear positive relationship between each Community Area’s TUI value and the average TUI of its neighbours, meaning that ridesourcing intensity is spatially clustered rather than randomly distributed: high-TUI areas tend to be near other high-TUI areas, and low-TUI areas tend to be near low-TUI areas.

### LISA
Is ridesourcing intensity spatially clustered, or randomly distributed across Chicago?
If high-TUI areas are near other high-TUI areas, then TUI has a spatial structure.

In [ ]:

lisa, analysis_gdf = compute_lisa(analysis_gdf, "tui_index")

In [ ]:
plot_lisa_map(analysis_gdf, 2024)

The LISA map localizes the pattern identified with Morans scatterplot: the blue High–High cluster identifies a core group of neighbouring areas with consistently high TUI, while the pink Low–Low clusters show areas where ridesourcing intensity is systematically low relative to surrounding areas; the red High–Low area is a spatial outlier with high TUI surrounded by lower-TUI neighbours.

## 4. TUI vs vulnerability

Pearson and Spearman correlations between mean TUI (2024) and HSVI, SDVI, and CCVI. Side-by-side choropleth maps compare spatial patterns.

In [ ]:
pearson, spearman, tests = compute_tui_correlations(analysis_gdf)

print("=== Pearson ===")
display(pearson.round(3))
print("\n=== Spearman ===")
display(spearman.round(3))

for _, row in tests.iterrows():
    print(
        f"TUI vs {row['indicator']}: "
        f"r={row['pearson_r']:.3f} (p={row['pearson_p']:.4f}), "
        f"rho={row['spearman_rho']:.3f} (p={row['spearman_p']:.4f})"
    )

ax = plot_tui_vulnerability_scatter(analysis_gdf, x="hsvi", year=2024)
save_chart(ax, "tui_vs_hsvi_2024")
plt.tight_layout()
plt.show()

ax = plot_tui_vulnerability_scatter(analysis_gdf, x="sdvi", year=2024)
save_chart(ax, "tui_vs_sdvi_2024")
plt.tight_layout()
plt.show()

In [ ]:
fig = plot_tui_vulnerability_maps(analysis_gdf, year=2024)
save_chart(fig, "tui_sdvi_maps_2024")
plt.show()

## 5. Weighted TUI (Financial Burden Index)

Estimates ridesharing financial burden per Community Area:

$$\text{total\_spend}_i = \text{n\_trips}_i \times \text{avg\_trip\_cost}_i$$
$$\text{rideshare\_spend\_pc}_i = \frac{\text{total\_spend}_i}{\text{population}_i}$$
$$\text{Weighted\_TUI}_i = \frac{\text{rideshare\_spend\_pc}_i}{\text{per\_capita\_income}_i}$$

Trip counts come from aggregated TNP data; a uniform mean trip cost proxy is applied because fare is not available in the count files. O'Hare (CA 76) excluded.

In [ ]:
median_trip_cost_2023_2024 = source_tnp_fares(url_2023_2024, "tnp_median_fares_2023_2024", force=False)

In [ ]:
weighted_tui_df = compute_weighted_tui(
    df,
    vulnerability,
    median_trip_cost_2023_2024,
    year=2024,   
)
weighted_tui_map = build_weighted_tui_map(chicago_community_areas, weighted_tui_df)

display(
    weighted_tui_df[
        ["community_area", "n_trips", "total_spend", "rideshare_spend_pc", "per_capita_income", "Weighted_TUI"]
    ].sort_values("Weighted_TUI", ascending=False).head(10)
)

ax = plot_weighted_tui_map(weighted_tui_map, year=2024)
save_chart(ax, "weighted_tui_2024")
plt.show()

## 6. Ridesharing travel time to the Loop

Mean `trip_seconds` for TNP trips with **dropoff Community Area 32** (Loop), grouped by pickup Community Area. O'Hare (CA 76) excluded from origins. Values are aggregated from the trip-level TNP API (monthly queries, cached locally).

In [ ]:
source_loop_travel_times(url_2023_2024, "tnp_loop_travel_times_2024", year=2024)
loop_travel_df = pd.read_csv("data/tnp_loop_travel_times_2024.csv")
loop_travel_df["community_area"] = loop_travel_df["community_area"].astype(int)

loop_travel_map = build_loop_travel_time_map(chicago_community_areas, loop_travel_df)
loop_travel_map["mean_trip_minutes"] = loop_travel_map["mean_trip_seconds"] / 60

display(
    loop_travel_df.assign(mean_trip_minutes=loop_travel_df["mean_trip_seconds"] / 60)
    .sort_values("mean_trip_seconds", ascending=False)
    .head(10)
)

ax = plot_loop_travel_time_map(loop_travel_map, year=2024)
save_chart(ax, "loop_travel_time_2024")
plt.show()

## 7. Transit Accessibility Index (supply density and service frequency)
It measures CTA transit supply within each Community Area on three dimensions: bus stop density, rail station density, and scheduled service frequency (weekday departures), each normalized by area size. Higher values mean better transit supply; the inverse standardized value gives the Transit Deficit Index.

### Transit Accessibility Index

For each Community Area \(i\) we compute three density measures, all normalized by area in km\(^2\):

- $BusDensity_i = \frac{N^{bus}_i}{Area_i}$ where $N^{bus}_i$ is the number of CTA bus stops
- $RailDensity_i = \frac{N^{rail}_i}{Area_i}$ where $N^{rail}_i$ is the number of CTA rail stations
- $FreqDensity_i = \frac{Departures_i}{Area_i}$ where $Departures_i$ is the total scheduled CTA departures on a representative weekday (from the GTFS feed)

Frequency upgrades the index from mere infrastructure *presence* to actual *service level*: a stop served every few minutes counts more than one served hourly.

Each measure is standardized (z-score). The **Transit Accessibility Index** is their average:

- $TransitAccess_i = \frac{z(BusDensity_i) + z(RailDensity_i) + z(FreqDensity_i)}{3}$

The **Transit Deficit Index** is the negative standardized accessibility:

- $TransitDeficit_i = -z(TransitAccess_i)$

Higher $TransitDeficit_i$ means weaker access to CTA service, lower means better-than-average accessibility.

In [ ]:
transit = compute_transit_accessibility(
    community_areas=chicago_community_areas,
    bus_path="data/CTA_BusStops/CTA_BusStops.shp",
    rail_path="data/CTA_-_'L'_(Rail)_Stations_20260630.geojson",
    gtfs_dir="data/google_transit",
)

transit.drop(columns="geometry").to_csv("data/transit_accessibility_index.csv", index=False)

In [ ]:
plot_transit_deficit(transit)

Red areas have higher transit deficit, meaning fewer CTA bus stops / rail stations per km² relative to the city average. Green areas have lower transit deficit, meaning better physical transit accessibility.

The strong green area corresponding to the Loop is expected because CTA rail and bus infrastructure is dense there.

In [ ]:
# least accessible areas
transit[
    ["community_area", "community_area_name", "n_bus_stops", "n_rail_stations",
     "departures_per_km2", "transit_deficit"]
].sort_values("transit_deficit", ascending=False).head(10)

In [ ]:
# most accessible areas
transit[
    ["community_area", "community_area_name", "n_bus_stops", "n_rail_stations",
     "departures_per_km2", "transit_deficit"]
].sort_values("transit_deficit").head(10)

## 8. OLS Regression

### Correlation Investigation

In [ ]:
tui_2024 = (
    tui[tui["year"] == 2024]
    .groupby("community_area", as_index=False)
    .agg(tui_index=("tui_index", "median"))
)

analysis_gdf = (
    chicago_community_areas
    .merge(tui_2024, on="community_area", how="left")
    .merge(transit.drop(columns="geometry"), on="community_area", how="left")
    .merge(vulnerability, on="community_area", how="left")
    .merge(loop_travel_map[["community_area", "mean_trip_seconds"]], on="community_area", how="left")
)

analysis_gdf = analysis_gdf[analysis_gdf["community_area"] != 76].copy()  # exclude O'Hare
analysis_gdf = analysis_gdf.dropna(subset=["tui_index", "transit_deficit", "hsvi", "mean_trip_seconds"])

In [ ]:
analysis_gdf[["tui_index", "transit_access", "transit_deficit", "sdvi", "hsvi", "ccvi_score","mean_trip_seconds"]].corr()

### OLS Regression

Predictors are z-standardized so coefficients are comparable and scale-related multicollinearity warnings are avoided. The dependent variable is `log1p(TUI)`.

In [ ]:
#skew analysis
analysis_gdf["tui_index"].dropna().plot(kind="kde")
analysis_gdf["tui_index"].dropna().hist(bins=30, density=True)
plt.xlabel("TUI Index")
plt.ylabel("Density")
plt.show()

print("Skewness:", analysis_gdf["tui_index"].skew())

In [ ]:
# take log of TUI index to reduce skewness
analysis_gdf["log_tui"] = np.log1p(analysis_gdf["tui_index"])

ols_predictors = ["transit_deficit", "hsvi", "mean_trip_seconds"]
for col in ols_predictors:
    analysis_gdf[f"{col}_z"] = zscore(analysis_gdf[col])

model = smf.ols(
    formula="log_tui ~ transit_deficit_z + hsvi_z + mean_trip_seconds_z",
    data=analysis_gdf,
).fit(cov_type="HC3")

print(model.summary())

In [ ]:
gdf_reg = analysis_gdf.dropna(subset=["log_tui", "transit_deficit", "hsvi", "mean_trip_seconds"]).copy()

w = Queen.from_dataframe(gdf_reg, use_index=False)
w.transform = "r"

gdf_reg["residuals"] = model.resid

moran_resid = Moran(gdf_reg["residuals"].values, w, permutations=999)

print("Moran's I residuals:", moran_resid.I)
print("p-value:", moran_resid.p_sim)

Platform mobility may not be compensating for public transport deficits; instead, it appears to be concentrated in already well-served urban areas, while vulnerable and peripheral areas remain less integrated into ridesourcing activity.

## 9. Spatial Model

OLS residuals show spatial autocorrelation, so we fit spatial models (SEM and SLAG) to account for geographic dependence.

In [ ]:
analysis_gdf["log_tui"] = np.log1p(analysis_gdf["tui_index"])

x_cols = [
    "transit_deficit",
    "hsvi",
    "mean_trip_seconds",
]

x_cols_reduced = [
    "transit_deficit",
    "hsvi",
]

### Spatial Error Model

In [ ]:
sem, sem_gdf, sem_w = run_spatial_error_model(
    analysis_gdf,
    y_col="log_tui",
    x_cols=x_cols
)

print(sem.summary)

In [ ]:
#check residuals for spatial autocorrelation
moran_sem = moran_residuals_spatial_model(sem, sem_w)
moran_sem_filtered = moran_residuals_spatial_model(sem, sem_w, filtered=True)


### Spatial Lag Model

In [ ]:
slag, slag_gdf, slag_w = run_spatial_lag_model(
    analysis_gdf,
    y_col="log_tui",
    x_cols=x_cols
)

print(slag.summary)

In [ ]:
moran_slag = moran_residuals_spatial_model(slag, slag_w)

In [ ]:
print("SEM AIC:", sem.aic)
print("SLAG AIC:", slag.aic)

Among spatial specifications, the **Spatial Error Model (SEM)** is preferred: it has the lower AIC and, after filtering the spatial error component, residual Moran's I is no longer significant (`I = 0.025`, `p = 0.321`).

The Spatial Lag Model (SLAG) also detects spatial dependence in TUI across neighbouring areas, but fits worse by AIC. We therefore interpret coefficients from the SEM.

Note: SEM and SLAG already z-standardize predictors internally, so the OLS scaling fix does not change these results. The SEM vs SLAG comparison uses the same regressors in both models, so their relative ranking is driven by the spatial specification, not by predictor scaling.

### Overall conclusion

The results do not support the initial hypothesis that ridesourcing intensity is highest in transit-poor or more vulnerable areas. Instead, TUI is concentrated in central, transit-rich, and less vulnerable areas, suggesting that ridesourcing in Chicago reinforces existing urban activity centres rather than compensating for public transport deficits.

### Limitations

- **TUI measures usage intensity, not dependency.** TUI normalizes pickups by *residents*, but pickups in central areas (Loop, Near North) are inflated by commuters, tourists and nightlife who do not live there. The TNP data has no rider residency, so this cannot be filtered out. The finding that ridesourcing concentrates in central, well-served areas therefore partly reflects where activity happens, not where residents depend on it.
- **Peripherality proxies overlap.** `transit_deficit` and `mean_trip_seconds` are correlated (r = 0.73). The reduced models (OLS and spatial) confirm that signs and conclusions are stable when only one is kept.
- **Vulnerability indicators are near-redundant.** `hsvi`, `sdvi` and `ccvi` are highly correlated (r = 0.86 to 0.96). We keep `hsvi` as the baseline; results are analogous with `sdvi`.
- **Cross-sectional design.** The analysis is for 2024 only and is associational, so it does not support causal claims.

- **Service frequency is from the current GTFS schedule** (2026), while trip data is 2024. CTA service levels are fairly stable over time, but a small temporal mismatch remains.

## Appendix A — Robustness and diagnostic checks

Supporting analyses that validate the main model: multicollinearity diagnostics (VIF) and reduced specifications dropping `mean_trip_seconds`. They confirm that the predictors carry independent information and that signs and significance are stable, reinforcing the main findings.

### Multicollinearity check (VIF)

VIF above 5 suggests problematic collinearity. We keep `hsvi` (not `sdvi`) as the vulnerability regressor because it is the project baseline and combines hardship with the external CCVI health index, while `sdvi` overlaps with the income-based Weighted TUI analysis later. `hsvi` and `sdvi` are nearly redundant (r = 0.96), so only one belongs in the model.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_cols = ["transit_deficit_z", "hsvi_z", "mean_trip_seconds_z"]
vif_df = pd.DataFrame({
    "variable": vif_cols,
    "VIF": [
        variance_inflation_factor(analysis_gdf[vif_cols].values, i)
        for i in range(len(vif_cols))
    ],
})
print(vif_df.to_string(index=False))

### Robustness: reduced model (no `mean_trip_seconds`)

`transit_deficit` and `mean_trip_seconds` are both proxies for peripherality (r = 0.73). This model keeps only `transit_deficit` to check that signs and significance are stable.

In [ ]:
model_reduced = smf.ols(
    formula="log_tui ~ transit_deficit_z + hsvi_z",
    data=analysis_gdf,
).fit(cov_type="HC3")

print(model_reduced.summary())

### Robustness: spatial models without `mean_trip_seconds`

Same predictors as the reduced OLS, to check the stability of `transit_deficit` and `hsvi`. Note that dropping `mean_trip_seconds` sharply worsens fit (AIC rises from ~74 to ~87-96): despite its correlation with `transit_deficit` (r = 0.73), travel time carries independent information, so the two are not redundant. In the reduced specification the preferred spatial model also flips from SEM to SLAG, confirming that the SEM-vs-SLAG choice depends on the regressor set. The main analysis therefore retains the full model.

In [ ]:
sem_reduced, _, _ = run_spatial_error_model(
    analysis_gdf,
    y_col="log_tui",
    x_cols=x_cols_reduced,
)
slag_reduced, _, _ = run_spatial_lag_model(
    analysis_gdf,
    y_col="log_tui",
    x_cols=x_cols_reduced,
)

print("Full model    — SEM AIC:", float(sem.aic), "| SLAG AIC:", float(slag.aic))
print("Reduced model — SEM AIC:", float(sem_reduced.aic), "| SLAG AIC:", float(slag_reduced.aic))
print()
print(sem_reduced.summary)
print()
print(slag_reduced.summary)